In [3]:
from pathlib import Path
from lxml import etree
import pandas as pd
import re
from typing import List, Dict, Any

# Regex razonables para pedimento (15 o 18 dígitos, con o sin espacios)
PED_REGEX_15 = re.compile(r'^\d{2}\s?\d{2}\s?\d{4}\s?\d{7}$')
PED_REGEX_18 = re.compile(r'^\d{2}\s?\d{2}\s?\d{4}\s?\d{7}\s?\d{3}$')
NS = {"cfdi": "http://www.sat.gob.mx/cfd/4", "tfd": "http://www.sat.gob.mx/TimbreFiscalDigital"}

def _get_attr(elem, *names, default=""):
    """Obtiene atributo sin importar mayúsc/minúsc en el nombre."""
    if elem is None:
        return default
    low = {k.lower(): v for k, v in elem.attrib.items()}
    for n in names:
        v = low.get(n.lower())
        if v is not None:
            return v
    return default

def _normalize_pedimento(s: str) -> str:
    """Normaliza a 'AA BB CCCC DDDDDDD [CCC]' cuando hay 15/18 dígitos."""
    digits = re.sub(r'\D', '', s or '')
    if len(digits) >= 15:
        a, b, c, d = digits[:2], digits[2:4], digits[4:8], digits[8:15]
        if len(digits) >= 18:
            e = digits[15:18]
            return f"{a} {b} {c} {d} {e}"
        return f"{a} {b} {c} {d}"
    return s or ''

def _unique_keep_order(seq: List[str]) -> List[str]:
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def _is_invoice(root) -> bool:
    """Solo facturas (TipoDeComprobante='I')."""
    tdc = _get_attr(root, "TipoDeComprobante", "tipodecomprobante").upper()
    return tdc == "I"

def _safe_date(root) -> str:
    """YYYY-MM-DD si trae hora."""
    f = _get_attr(root, "Fecha", "fecha")
    return f[:10] if f and len(f) >= 10 else f

def _collect_document_level_peds(root) -> List[str]:
    """Busca pedimentos en cualquier InformacionAduanera del documento (fallback)."""
    peds = root.xpath('.//*[local-name()="InformacionAduanera"]/@NumeroPedimento')
    peds = [_normalize_pedimento(p) for p in peds if p]
    peds = [p for p in peds if PED_REGEX_15.match(p) or PED_REGEX_18.match(p)]
    return _unique_keep_order(peds)


In [ ]:
def parse_cfdi_folder_concept_pedimento_only(folder: str) -> pd.DataFrame:
    base = Path(folder)
    if not base.exists():
        raise FileNotFoundError(f"No existe la carpeta: {base}")
    
    rows = []
    for xml_path in base.rglob("*.xml"):
        try:
            root = etree.parse(str(xml_path), etree.XMLParser(recover=True, huge_tree=True)).getroot()
        except Exception:
            continue  # XML ilegible

        # Solo Comprobante CFDI 4.0 y facturas (I)
        if root.tag.endswith("Comprobante") and _is_invoice(root):
            fecha = _safe_date(root)
            emisor = root.find("cfdi:Emisor", NS)
            emisor_rfc = _get_attr(emisor, "Rfc")
            emisor_nombre = _get_attr(emisor, "Nombre")

            # NUEVO: serie, folio y uuid
            serie = _get_attr(root, "Serie", "serie")
            folio = _get_attr(root, "Folio", "folio")
            tfd = root.find(".//tfd:TimbreFiscalDigital", NS)
            uuid = _get_attr(tfd, "UUID", "uuid")

            # Recorre conceptos
            for c in root.findall(".//cfdi:Conceptos/cfdi:Concepto", NS):
                # Pedimentos SOLO del concepto (no fallback a nivel documento)
                peds = [ia.get("NumeroPedimento", "").strip()
                        for ia in c.findall("./cfdi:InformacionAduanera", NS)]
                pedimento = " | ".join([p for p in peds if p])  # puede quedar ""

                rows.append({
                    "archivo": xml_path.name,
                    "serie": serie,
                    "folio": folio,
                    "uuid": uuid,
                    "fecha": fecha,
                    "emisor_rfc": emisor_rfc,
                    "emisor_nombre": emisor_nombre,
                    "concepto_clave_prod_serv": c.get("ClaveProdServ", ""),
                    "concepto_no_identificacion": c.get("NoIdentificacion", ""),
                    "concepto_cantidad": c.get("Cantidad", ""),
                    "concepto_clave_unidad": c.get("ClaveUnidad", ""),
                    "concepto_unidad": c.get("Unidad", ""),
                    "concepto_descripcion": c.get("Descripcion", ""),
                    "concepto_valor_unitario": c.get("ValorUnitario", ""),
                    "concepto_importe": c.get("Importe", ""),
                    "pedimento": pedimento,  # vacío si el concepto no trae InformacionAduanera
                })
    
    df = pd.DataFrame(rows)
    df['concepto_no_identificacion'] = (
        df['concepto_no_identificacion']
        .astype(str)          # asegura que todo es texto
        .str.lstrip('0')      # elimina ceros a la izquierda
    )
    if not df.empty:
        df = df.reindex(columns=[
            "archivo","serie","folio","uuid","fecha","emisor_rfc","emisor_nombre",
            "concepto_clave_prod_serv","concepto_no_identificacion","concepto_cantidad",
            "concepto_clave_unidad","concepto_unidad","concepto_descripcion",
            "concepto_valor_unitario","concepto_importe","pedimento"
        ])
    return df


# === Ejecuta aquí cambiando la ruta ===
RUTA = "CFDIS"  # <-- CAMBIA ESTA RUTA
df = parse_cfdi_folder_concept_pedimento_only(RUTA)

df.to_excel('Reporte Compras 2023-2025.xlsx')


In [ ]:
import pandas as pd
# Solo en Odoo

orders = self.env['sale.order'].search([('id','in',[84194, 84193, 84192, 84177, 84140])])
move_lines = orders.mapped('picking_ids').mapped('move_line_ids_without_package')

rows = []
for line in move_lines:
    lot = line.lot_id
    rows.append({
        "quant_id": line.id,  # id de stock.move.line
        "lot_name": lot.name if lot else "",  # nombre del lote
        "lot_id": lot.id if lot else "",      # id del lote
        "pedimento": lot.pedimento if lot and hasattr(lot, "pedimento") else "",  # campo adicional
        "concepto_clave_prod_serv": line.product_id.default_code or "",
        "producto_nombre": line.product_id.display_name,
        "concepto_cantidad": line.qty_done or line.reserved_uom_qty,
        "ubicacion": line.location_id.display_name,
    })

df = pd.DataFrame(rows)

df.to_excel('/mnt/extra-addons/Compras Quant.xlsx')


In [ ]:
#Leer reporte

df_stock_quant = pd.read_excel('/mnt/extra-addons/Compras Quant.xlsx')

cfdi_compras = pd.read_excel('/mnt/extra-addons/Reporte Compras 2023-2025.xlsx')


# --- Función de cálculo ---
def calcular_campo(row, df_ref):
    """
    row: fila de df_stock_quant
    df_ref: dataframe cfdi_compras
    """
    # Ejemplo: buscar coincidencia por 'producto' y traer total de compras
    producto = row['concepto_clave_prod_serv']   # ajusta el campo que sirve como llave
    compras = df_ref.loc[df_ref['concepto_clave_prod_serv'] == producto, 'concepto_cantidad'].sum()
    
    return compras

# --- Aplicar la función ---
df_stock_quant['compras_totales'] = df_stock_quant.apply(
    lambda r: calcular_campo(r, cfdi_compras), axis=1
)
